# Charge sharpening: benchmark against Agrawal et al. 2024, Fig. 7

Experiment `experiments/charge_sharpening/charge_sharpening.py` (issues #137 and #142). It reproduces Fig. 7 of U. Agrawal *et al.*, “Observing quantum measurement collapse as a learnability phase transition”, PRX **14**, 041012 (2024), at the paper's size: 6 system qubits plus the ancilla `R`, that is `N = 7` total qubits here. The paper counts the system sites only, so its curves named `L = 6` and `L = 20` are shown as `N = 7` and `N = 21`.

The protocol, configured by `conf/charge_sharpening.yaml`:

1. initial state, Eq. (B1): `(|0>_R |010101> + |1>_R |010100>) / sqrt(2)`, the ancilla entangled with the charge 3 or 2 of the system, then scrambled by 5 layers of all-to-all random U(1) gates;
2. `t = Ns / 2 = 3` brickwork layers of random U(1) gates (`Ns = N - 1 = 6` system qubits; one layer = one full period, even then odd bonds), one fixed circuit instance per seed as in the paper;
3. between two layers (2 rounds), each system qubit is weakly measured with strength `gamma`: `gamma = 0` learns nothing, `gamma = 1` is projective;
4. the entanglement entropy `S_R` of the ancilla (in nats, as in the paper) is kept for each of the `L` trajectories.

The observables of Fig. 7 are the mean of `S_R` over the trajectories, its variance `mu_2` and the Binder ratio `B = 1 - mu_4 / (3 mu_2**2)`. As `gamma` grows, the measurements learn the charge of the system and `S_R` falls from `ln 2` to 0: the charge “sharpens”. The variance peaks where the trajectories split between the two behaviors, and the Binder curves of different sizes cross there.

This notebook launches the sweep with `experiments/charge_sharpening/charge_sharpening.sh`, reads back the folders `outputs/charge_sharpening_fig7/<date_time>/N<N>_seed<seed>_g<gamma>/`, draws the three panels over the curves digitized from the paper (`experiments/charge_sharpening/reference/`), then shows the distribution of `S_R` over the trajectories, which the variance and the Binder ratio summarize.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_charge_sharpening_notebook.py` does for a small sweep). Notation, the same in every notebook: `N` is the total number of qubits, ancilla included, and `L` the number of samples per run (`samples.L` in the config). Each sweep goes to this notebook's own folder, `outputs/charge_sharpening_fig7/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one.

With the default values (`L = 10000` trajectories as in the paper, 3 circuit instances, 11 values of `gamma`), the sweep takes a few minutes. Each seed is a different circuit instance: the band around our points is their spread.

In [ ]:
L = os.environ.get("L", "10000")  # trajectories per run (samples.L)
N_VALUES = os.environ.get("N_VALUES", "7")  # total qubits N: 6 system + R
SEEDS = os.environ.get("SEEDS", "0 1 2")  # one circuit instance per seed
GAMMA_VALUES = os.environ.get(
    "GAMMA_VALUES", "0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0"
)
FIXED_INSTANCE = os.environ.get("FIXED_INSTANCE", "true")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151): no other notebook writes
# there, so the cells below never read another notebook's sweep.
EXPERIMENT = "charge_sharpening_fig7"

if RUN:
    # Create the sweep folder here and hand it to the launcher (OUT_DIR):
    # the cells below read this sweep and no other.
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "L": L,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "GAMMA_VALUES": GAMMA_VALUES,
        "FIXED_INSTANCE": FIXED_INSTANCE,
        "OUT_DIR": str(sweep_dir),
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(GAMMA_VALUES.split())
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/charge_sharpening/charge_sharpening.sh"],
        total=n_runs,
        env=env,
        show=("N", "seed", "gamma"),
    )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the observables

One row per `(N, gamma)`, averaged over the seeds (circuit instances): the mean ± the standard error between instances.

In [ ]:
import importlib.util

from QELM.utils import load_sweep, summarize

# The plotting script of the experiment is not a package: import it by path.
spec = importlib.util.spec_from_file_location(
    "plot_charge_sharpening",
    Path("experiments/charge_sharpening/plot_charge_sharpening.py"),
)
plot_cs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_cs)

METRICS = [panel[0] for panel in plot_cs.PANELS]  # S_R_mean, S_R_var, binder
rows = load_sweep(sweep_dir, keys=plot_cs.KEYS)
table = summarize(rows, by=("N", "gamma"), metrics=METRICS)
print(f"{len(rows)} runs")
print(f"{'N':>3} {'gamma':>6} {'runs':>4} " + " ".join(f"{m:>20}" for m in METRICS))
for r in table:
    values = " ".join(
        f"{r[f'{m}_mean']:>9.4f} ± {r[f'{m}_sem']:<8.4f}" for m in METRICS
    )
    print(f"{r['N']:>3} {r['gamma']:>6.2f} {r['n_runs']:>4} {values}")

## Fig. 7: our points over the paper's

Three panels, as in Fig. 7: `S_R` (nats), its variance (nats²) and the Binder ratio, against `gamma`. Our points are the means over the instances, with the standard error as bars and the spread between instances as a band; with a single seed, the bar is the bootstrap error over the trajectories. The dashed curves are the data digitized from the paper, `N = 7` (to compare) and `N = 21` (to see the size effect).

The Binder ratio is drawn only where the variance exceeds `1e-3`: elsewhere `S_R` sticks to `ln 2` or to 0, and a few rare trajectories make `B` very negative. Below the plot, the deviation from the paper's `N = 7` curve, interpolated at its points.

The figure, the data it shows and the comparison are saved in the sweep folder: `charge_sharpening.png`, `charge_sharpening.json` and `comparison.json`.

In [ ]:
import json

import matplotlib.pyplot as plt

from QELM.utils.plots import print_comparison, save_figure

fig, comparison = plot_cs.draw_sweep(sweep_dir)
figure_path = save_figure(fig, sweep_dir, plot_cs.FIGURE_NAME)
(sweep_dir / "comparison.json").write_text(json.dumps(comparison, indent=2) + "\n")
print_comparison(comparison)
print(f"figure: {figure_path}")
plt.show()

## Distribution of `S_R` over the trajectories

The mean, the variance and the Binder ratio summarize this distribution. At weak `gamma`, almost every trajectory keeps `S_R ≈ ln 2`; at strong `gamma`, almost every one has purified it (`S_R ≈ 0`). Near the transition, the distribution spreads between the two: this is where the variance peaks and `B` is positive. One instance (the first seed) and the largest size of the sweep.

The histograms and their counts are saved in the sweep folder as `S_R_distribution.png` and `S_R_distribution.json`.

In [ ]:
import numpy as np

from QELM.utils.plots import COLORS, subplots

N_hist = max(int(r["N"]) for r in rows)
seed_hist = min(int(r["seed"]) for r in rows if int(r["N"]) == N_hist)
runs = sorted(
    (r for r in rows if int(r["N"]) == N_hist and int(r["seed"]) == seed_hist),
    key=lambda r: r["gamma"],
)
# Up to 5 values of gamma, spread over the sweep.
picks = [
    runs[k] for k in np.unique(np.linspace(0, len(runs) - 1, 5).round().astype(int))
]
hist_fig, hist_axes = subplots(
    1, len(picks), figsize=(3.8 * len(picks), 4), sharey=True, squeeze=False
)
bins = np.linspace(0, np.log(2), 31)
for ax, r in zip(hist_axes[0], picks, strict=True):
    S_R = np.load(sweep_dir / r["run"] / "charge_sharpening.npz")["S_R"]
    ax.hist(
        np.clip(S_R, 0, np.log(2)),
        bins=bins,
        color=COLORS[0],
        label=f"gamma = {r['gamma']:.2f}",
    )
    ax.set(title=f"gamma = {r['gamma']:.2f}", xlabel="S_R (nats)")
    ax.set_yscale("log")
hist_axes[0, 0].set_ylabel("trajectories")
hist_fig.suptitle(f"Distribution of S_R, N = {N_hist}, seed {seed_hist}")
hist_path = save_figure(hist_fig, sweep_dir, "S_R_distribution")
print(f"figure: {hist_path}")
plt.show()